In [ ]:

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from bs4 import BeautifulSoup
import pandas as pd
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
from pathlib import Path

In [ ]:

def get_team_standing(season):
    service = Service(executable_path=ChromeDriverManager().install())

    chrome_options = Options()

    chrome_options.add_experimental_option("detach", True)
    chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])


    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    driver.get(f"https://www.mlb.com/standings/mlb/{season}")

    data = []

    table = WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".tablestyle__StyledTable-sc-wsl6eq-0.cprRUf.StandingsTablestyle__DataTableWrapper-sc-1l6jbjt-1.kCeyFZ.auto-scroller"))
    )


    header_row = table.find_element(By.TAG_NAME, "thead")
    headers = [th.text.strip() for th in header_row.find_elements(By.TAG_NAME, "th")]

    rows = table.find_elements(By.TAG_NAME, "tr")

    for row in rows:
        cols = row.find_elements(By.XPATH, ".//th | .//td")
        if cols:
            data.append([col.text.strip() for col in cols])
    
    df = pd.DataFrame(data=data, columns=headers)

    df = df.iloc[1:, :]
    df.TEAM = df.TEAM.str.split("\n").str[0]
    
    df.W = df.W.astype("int")
    df.L = df.L.astype("int")
    df['win_rate'] = df.W / (df.W + df.L) 
    
    df.to_csv(Path.cwd() / "data" / str(season) / "team_standings.csv")

In [8]:
for season in range(2023, 2012, -1):
    get_team_standing(season)